# TradingAgents 파이프라인 — 단계별 sub-graph 노트북

원본 `setup.py`의 단일 풀그래프를 **5개 sub-graph**로 쪼개 각 셀에서 따로 실행한다.
모든 sub-graph는 같은 `AgentState`를 입력/출력으로 쓰므로 셀 간에 전역 `state` 딕셔너리만 넘기면 된다.

```
Market → Sentiment → News → Fundamentals → Research(debate+manager) → Trader → Risk(3자 debate + PM)
```

**전제**
- 사용 LLM 제공자의 API 키가 환경에 설정돼 있어야 함 (`OPENAI_API_KEY` 등)
- 한국 종목 펀더멘털 보강은 `DART_API_KEY` (없어도 동작)
- 비용 주의: 셀을 다시 실행하면 그 단계의 LLM 호출이 다시 일어남


## 1. Setup — config, LLM, 공통 컴포넌트

In [ ]:
import os
from tradingagents.default_config import DEFAULT_CONFIG
from tradingagents.dataflows.config import set_config
from tradingagents.llm_clients import create_llm_client
from tradingagents.agents.utils.memory import TradingMemoryLog
from tradingagents.graph.propagation import Propagator
from tradingagents.graph.conditional_logic import ConditionalLogic
from tradingagents.graph.signal_processing import SignalProcessor

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
RUNTIME_DIR = os.path.join(PROJECT_DIR, ".runtime")

CONFIG = dict(DEFAULT_CONFIG)
# 로컬 Ollama (run_gemma.py와 동일 설정). 다른 제공자로 가려면 아래 블록을 교체.
CONFIG["llm_provider"]            = "ollama"
CONFIG["backend_url"]             = "http://localhost:11434/v1"
CONFIG["deep_think_llm"]          = "gemma4:26b-a4b"
CONFIG["quick_think_llm"]         = "gemma4:26b-a4b"
CONFIG["max_debate_rounds"]       = 1
CONFIG["max_risk_discuss_rounds"] = 1
CONFIG["output_language"]         = "Korean"
CONFIG["results_dir"]             = os.path.join(RUNTIME_DIR, "logs")
CONFIG["data_cache_dir"]          = os.path.join(RUNTIME_DIR, "cache")
CONFIG["memory_log_path"]         = os.path.join(RUNTIME_DIR, "memory", "trading_memory.md")
# analyst_mode="auto"가 gemma 패턴을 감지해 prefetch 모드로 자동 전환 (tool_calling 미지원 모델 대응).

# --- 대안: OpenAI / Anthropic / Gemini로 갈 때 ---
# CONFIG["llm_provider"]    = "openai"
# CONFIG["backend_url"]     = None
# CONFIG["deep_think_llm"]  = "gpt-5.4"
# CONFIG["quick_think_llm"] = "gpt-5.4-mini"
# # 환경에 OPENAI_API_KEY / ANTHROPIC_API_KEY / GOOGLE_API_KEY 가 있어야 함.

set_config(CONFIG)

deep_client = create_llm_client(
    provider=CONFIG["llm_provider"],
    model=CONFIG["deep_think_llm"],
    base_url=CONFIG.get("backend_url"),
)
quick_client = create_llm_client(
    provider=CONFIG["llm_provider"],
    model=CONFIG["quick_think_llm"],
    base_url=CONFIG.get("backend_url"),
)
deep_llm = deep_client.get_llm()
quick_llm = quick_client.get_llm()

conditional_logic = ConditionalLogic(
    max_debate_rounds=CONFIG["max_debate_rounds"],
    max_risk_discuss_rounds=CONFIG["max_risk_discuss_rounds"],
)
propagator = Propagator(max_recur_limit=CONFIG.get("max_recur_limit", 100))
signal_processor = SignalProcessor(quick_llm)
memory_log = TradingMemoryLog(CONFIG)

INVOKE_CFG = {"recursion_limit": CONFIG.get("max_recur_limit", 100)}
print(f"provider={CONFIG['llm_provider']}, model={CONFIG['quick_think_llm']}, backend={CONFIG.get('backend_url')}")


## 2. Tool node + sub-graph 빌더 정의

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

from tradingagents.agents.utils.agent_states import AgentState
from tradingagents.agents.utils.agent_utils import (
    get_stock_data, get_indicators,
    get_fundamentals, get_balance_sheet, get_cashflow, get_income_statement,
    get_news, get_global_news, get_insider_transactions,
)
from tradingagents.agents import (
    create_market_analyst, create_sentiment_analyst,
    create_news_analyst, create_fundamentals_analyst,
    create_bull_researcher, create_bear_researcher, create_research_manager,
    create_trader,
    create_aggressive_debator, create_conservative_debator, create_neutral_debator,
    create_portfolio_manager,
    create_msg_delete,
)
from tradingagents.graph.analyst_execution import ANALYST_NODE_SPECS

TOOL_NODES = {
    "market":       ToolNode([get_stock_data, get_indicators]),
    "social":       ToolNode([get_news]),
    "news":         ToolNode([get_news, get_global_news, get_insider_transactions]),
    "fundamentals": ToolNode([get_fundamentals, get_balance_sheet, get_cashflow, get_income_statement]),
}

ANALYST_FACTORIES = {
    "market":       lambda: create_market_analyst(quick_llm),
    "social":       lambda: create_sentiment_analyst(quick_llm),
    "news":         lambda: create_news_analyst(quick_llm),
    "fundamentals": lambda: create_fundamentals_analyst(quick_llm),
}

def build_analyst_subgraph(key: str):
    """analyst ↔ tool ReAct 루프 + Msg Clear까지를 한 sub-graph로 묶는다."""
    spec = ANALYST_NODE_SPECS[key]
    g = StateGraph(AgentState)
    g.add_node(spec.agent_node, ANALYST_FACTORIES[key]())
    g.add_node(spec.tool_node, TOOL_NODES[key])
    g.add_node(spec.clear_node, create_msg_delete())
    g.add_edge(START, spec.agent_node)
    g.add_conditional_edges(
        spec.agent_node,
        getattr(conditional_logic, f"should_continue_{key}"),
        [spec.tool_node, spec.clear_node],
    )
    g.add_edge(spec.tool_node, spec.agent_node)
    g.add_edge(spec.clear_node, END)
    return g.compile()

def build_research_subgraph():
    """Bull ⇄ Bear debate → Research Manager."""
    g = StateGraph(AgentState)
    g.add_node("Bull Researcher",   create_bull_researcher(quick_llm))
    g.add_node("Bear Researcher",   create_bear_researcher(quick_llm))
    g.add_node("Research Manager",  create_research_manager(deep_llm))
    g.add_edge(START, "Bull Researcher")
    g.add_conditional_edges("Bull Researcher", conditional_logic.should_continue_debate, {
        "Bear Researcher": "Bear Researcher",
        "Research Manager": "Research Manager",
    })
    g.add_conditional_edges("Bear Researcher", conditional_logic.should_continue_debate, {
        "Bull Researcher": "Bull Researcher",
        "Research Manager": "Research Manager",
    })
    g.add_edge("Research Manager", END)
    return g.compile()

def build_trader_subgraph():
    g = StateGraph(AgentState)
    g.add_node("Trader", create_trader(quick_llm))
    g.add_edge(START, "Trader")
    g.add_edge("Trader", END)
    return g.compile()

def build_risk_subgraph():
    """Aggressive ⇄ Conservative ⇄ Neutral 3자 debate → Portfolio Manager."""
    g = StateGraph(AgentState)
    g.add_node("Aggressive Analyst",   create_aggressive_debator(quick_llm))
    g.add_node("Conservative Analyst", create_conservative_debator(quick_llm))
    g.add_node("Neutral Analyst",      create_neutral_debator(quick_llm))
    g.add_node("Portfolio Manager",    create_portfolio_manager(deep_llm))
    g.add_edge(START, "Aggressive Analyst")
    g.add_conditional_edges("Aggressive Analyst", conditional_logic.should_continue_risk_analysis, {
        "Conservative Analyst": "Conservative Analyst",
        "Portfolio Manager":    "Portfolio Manager",
    })
    g.add_conditional_edges("Conservative Analyst", conditional_logic.should_continue_risk_analysis, {
        "Neutral Analyst":   "Neutral Analyst",
        "Portfolio Manager": "Portfolio Manager",
    })
    g.add_conditional_edges("Neutral Analyst", conditional_logic.should_continue_risk_analysis, {
        "Aggressive Analyst": "Aggressive Analyst",
        "Portfolio Manager":  "Portfolio Manager",
    })
    g.add_edge("Portfolio Manager", END)
    return g.compile()

print("sub-graph builders ready")


## 3. 초기 상태 — 티커/날짜 지정

In [ ]:
TICKER = "005930.KS"   # 예: AAPL, NVDA, 005930.KS (삼성전자), BTC-USD
DATE   = "2026-05-22"
ASSET  = "stock"        # crypto 분석은 "crypto"

state = propagator.create_initial_state(
    company_name=TICKER,
    trade_date=DATE,
    asset_type=ASSET,
    past_context=memory_log.get_past_context(TICKER),
)
print({k: type(v).__name__ for k, v in state.items()})


## 4. Analyst 단계 (4개 sub-graph)

각 셀은 해당 analyst의 ReAct 루프(에이전트 ↔ 툴)와 메시지 클리어까지 자기 sub-graph 안에서 처리한다.
report 필드(`market_report` 등)에 결과가 채워지면 그 단계는 성공.


In [ ]:
market_graph = build_analyst_subgraph("market")
state = market_graph.invoke(state, config=INVOKE_CFG)
print("=== market_report ===\n", state["market_report"][:1000])


In [ ]:
sentiment_graph = build_analyst_subgraph("social")
state = sentiment_graph.invoke(state, config=INVOKE_CFG)
print("=== sentiment_report ===\n", state["sentiment_report"][:1000])


In [ ]:
news_graph = build_analyst_subgraph("news")
state = news_graph.invoke(state, config=INVOKE_CFG)
print("=== news_report ===\n", state["news_report"][:1000])


In [ ]:
fundamentals_graph = build_analyst_subgraph("fundamentals")
state = fundamentals_graph.invoke(state, config=INVOKE_CFG)
print("=== fundamentals_report ===\n", state["fundamentals_report"][:1000])


## 5. Research debate — Bull ⇄ Bear → Research Manager

`max_debate_rounds`만큼 왕복 후 Research Manager가 `investment_plan`을 산출.


In [ ]:
research_graph = build_research_subgraph()
state = research_graph.invoke(state, config=INVOKE_CFG)
print("=== investment_plan ===\n", state["investment_plan"][:1500])


## 6. Trader — analyst 보고서 + investment_plan 기반 트레이딩 플랜

In [ ]:
trader_graph = build_trader_subgraph()
state = trader_graph.invoke(state, config=INVOKE_CFG)
print("=== trader_investment_plan ===\n", state["trader_investment_plan"][:1500])


## 7. Risk debate — Aggressive ⇄ Conservative ⇄ Neutral → Portfolio Manager

`max_risk_discuss_rounds × 3` 한도까지 3자 라운드를 돌고 Portfolio Manager가 `final_trade_decision` 산출.


In [ ]:
risk_graph = build_risk_subgraph()
state = risk_graph.invoke(state, config=INVOKE_CFG)
print("=== final_trade_decision ===\n", state["final_trade_decision"])


## 8. Signal 추출 — BUY / HOLD / SELL

In [ ]:
signal = signal_processor.process_signal(state["final_trade_decision"])
print("signal:", signal)


## 9. (선택) 특정 단계만 재실행

각 sub-graph는 입력 `state`만 있으면 독립적으로 다시 돌릴 수 있다. 예시:

```python
# Market analyst만 재실행 — state["market_report"]를 ""로 비우고 다시 호출
state["market_report"] = ""
state["messages"] = []
state = market_graph.invoke(state, config=INVOKE_CFG)
```

debate는 라운드 카운터(`state["investment_debate_state"]["count"]`)를 0으로 리셋해야 한도 체크가 다시 처음부터 동작한다.
